# Kjarni Embeddings

[Kjarni](https://kjarni.ai) runs embedding models in your own process, with no PyTorch, no ONNX Runtime, no GPU and no server, and no API key. It produces the same vectors as `sentence-transformers` for the same model. The model downloads once on first use and is cached; after that everything works offline.

If you're opening this notebook on Colab, you will probably need to install LlamaIndex 🦙.

In [1]:
%pip install -qU llama-index-embeddings-kjarni llama-index

Note: you may need to restart the kernel to use updated packages.


## Embeddings

In [2]:
from llama_index.embeddings.kjarni import KjarniEmbedding

embed_model = KjarniEmbedding(model_name="minilm-l6-v2")

embedding = embed_model.get_text_embedding("Kjarni runs embedding models locally.")
print(len(embedding))
print(embedding[:5])

384
[0.013499433174729347, -0.04745718464255333, 0.01434510201215744, -0.044113222509622574, 0.06950274854898453]


## Batches and queries

In [3]:
texts = ["Reykjavik is the capital of Iceland.", "Sourdough needs a starter."]
vectors = embed_model.get_text_embedding_batch(texts)
query = embed_model.get_query_embedding("What is Iceland's capital city?")

scores = [sum(q * v for q, v in zip(query, vector)) for vector in vectors]
for text, score in zip(texts, scores):
    print(f"{score:.3f}  {text}")

0.827  Reykjavik is the capital of Iceland.
0.088  Sourdough needs a starter.


## In an index

In [4]:
from llama_index.core import Document, Settings, VectorStoreIndex

Settings.embed_model = embed_model

documents = [
    Document(text="Kjarni is a native inference engine with bindings for C#, C++, Python, Go, Rust and the browser."),
    Document(text="A cross-encoder reranker reads the query and each passage together, and scores them more precisely than vectors alone."),
    Document(text="Whisper transcribes speech to text, and detects the language spoken."),
]
index = VectorStoreIndex.from_documents(documents)
for node in index.as_retriever(similarity_top_k=1).retrieve("Can I use it from C# or Go?"):
    print(f"{node.score:.3f}  {node.node.get_content()}")

0.308  Kjarni is a native inference engine with bindings for C#, C++, Python, Go, Rust and the browser.


## Other models

`model_name` takes any Kjarni embedding model: `minilm-l6-v2` (384 dimensions, the default), `mpnet-base-v2` (768), `nomic-embed-text` (768, long inputs) or `bge-m3` (1024, multilingual). For models trained with instruction prefixes, set `query_prefix` and `text_prefix`. See the [Kjarni documentation](https://kjarni.ai/docs/).